In [1]:
import scanpy as sc
import sys
import reference_based_harmonizing

import data to be harmonized:

In [3]:
adata = sc.read_h5ad("/storage/groups/ml01/workspace/jan.domke/Data/data_to_harmonize.h5ad")

Original cell type labeling (as provided by dataset providers) will now be translated to the current version of the cell type ontology, consisting of 5 levels.

First, import the .csv that contains the translations:

In [4]:
harmonizing_df = reference_based_harmonizing.load_harmonizing_table(
    f"LCA_cell_type_reference_mapping_20211103.csv"
)

Create a dataframe that contains each cell type name from the consensus ontology as indices, with their matching annotations at the other levels. This will simplify mapping:

In [5]:
consensus_df = reference_based_harmonizing.create_consensus_table(harmonizing_df)

since the Banovich_Kropski_2020 and Misharin_Budinger_2018 study have different cell type annotations/translations for disease than for healthy, and since harmonization is done based on study name, we will temporarily rename the studies here:

In [6]:
study_to_harmstudy = {st: st for st in adata.obs.study.unique()}
study_to_harmstudy["Banovich_Kropski_2020"] = "Banovich_Kropski_2020_disease"
study_to_harmstudy["Misharin_Budinger_2018"] = "Misharin_Budinger_2018_disease"
# also store reverse, for translation back to original:
harmstudy_to_study = {val: key for key, val in study_to_harmstudy.items()}

In [7]:
adata.obs.study = adata.obs.study.map(study_to_harmstudy)

create a dataframe that for each original celltype annotation (from all datasets pooled) provides the translation to the consensus ontology at all levels:

In [8]:
celltype_translation_df = (
    reference_based_harmonizing.create_orig_ann_to_consensus_translation_df(
        adata, consensus_df, harmonizing_df, verbose=False
    )
)

now translate the original annotations to the consensus in your AnnData. (Note that these will include "forward-propagated" labels, i.e. if a cell is annotated as "Immune" only, it will have label "1_Immune" for level 2-5. We will create a "clean" annotation without forward-propagation further below.)

In [9]:
adata = reference_based_harmonizing.consensus_annotate_anndata(
    adata, celltype_translation_df, verbose=True
)

now remove "unicorns and artifacts" (cells annotated as low quality, doublets etc.), if they're there:

In [10]:
print(
    "Number of unicorns and artifact cells to remove:",
    sum(adata.obs.ann_level_1 == "Unicorns and artifacts"),
)

Number of unicorns and artifact cells to remove: 0


In [11]:
adata.shape

(776269, 93512)

In [12]:
adata = adata[adata.obs.ann_level_1 != "Unicorns and artifacts", :].copy()

In [13]:
adata.shape

(776269, 93512)

add "clean" annotations (with "None" for cells that do not have annotation at that level):

In [14]:
adata = reference_based_harmonizing.add_clean_annotation(adata)

convert back study names:

In [15]:
adata.obs.study = adata.obs.study.map(harmstudy_to_study)

check:

In [16]:
sorted(adata.obs.study.unique())

['Banovich_Kropski_2020',
 'Duong_lungMAP_unpubl',
 'Eils_2020',
 'Kaminski_2020',
 'Meyer_2021',
 'Misharin_Budinger_2018',
 'Sheppard_2020',
 'Sun_2020']

split in IPF dataset and nucleus dataset

In [27]:
nucleus_data = adata[adata.obs["cells_or_nuclei"]=="nuclei",:].copy()

In [28]:
ipf_data = adata[adata.obs["cells_or_nuclei"]!="nuclei",:].copy()

In [40]:
nucleus_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/nucleus/harmonized_nucleus_data.h5ad")

In [41]:
ipf_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/harmonized_ipf_data.h5ad")

/home/icb/jan.domke/miniconda3/envs/scArches/lib/python3.7/site-packages/anndata/_core/anndata.py:1220: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'ann_level_1' as categorical
/home/icb/jan.domke/miniconda3/envs/scArches/lib/python3.7/site-packages/anndata/_core/anndata.py:1220: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'ann_level_2' as categorical
/home/icb/jan.domke/miniconda3/envs/scArches/lib/python3.7/site-packages/anndata/_core/anndata.py:1220: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder